Version: 1.1
Updated: Aug 16

# Step 4B — Document-Level Evaluation on Colab

Runs `evaluate_document.py` against **D-Wikipedia's own test split** with D-SARI, LENS and
BERTScore — the document-appropriate benchmark that the ASSET-based Step 4 is not.

**Why Colab:** measured on Apple Silicon (MPS), generation runs ~8.4 s/document, which puts
a 500-document run at 2–5 hours and the full 8,000-document split past 37 hours. A T4 does
the same work in a fraction of that.

**What this produces:** a timestamped JSON carrying the scores *plus* full provenance —
model id, resolved checkpoint, weights mtime/size, git commit, device, decoding parameters,
sample size and seed. That file is what fills row **D2** in `research/results/RESULTS.md`.

Results are written directly to Google Drive as they are produced, so a dropped
connection costs minutes rather than the whole run (§2).

**Prerequisite:** Runtime → Change runtime type → **T4 GPU** (or better).

## 1. Confirm the GPU

In [ ]:
!nvidia-smi
import torch

print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun.")

## 2. Mount Google Drive

Colab's local disk dies with the runtime — including when the browser tab disconnects — so
anything written only to `/content` is gone before it can be downloaded. Mounting Drive and
pointing `--outdir` at it means every artifact lands somewhere permanent *as it is written*,
rather than at the end of a run that may not reach its end.

This matters more than it sounds. `evaluate_document.py` flushes partial generations to
`OUTDIR` every few batches and rewrites the result JSON after each scoring stage. Written to
Drive, those checkpoints outlive the runtime, so a rerun in a **fresh session** picks up the
generation where the dead one stopped. Written to local disk, they die with it and the run
starts from zero.

Same pattern as `train_sentence_and_document_pipeline.ipynb`'s Drive cell.

In [ ]:
# Colab only: mount Drive so generations/results under OUTDIR survive the session.
from google.colab import drive

drive.mount("/content/drive")

import os

# Artifacts are written here as they are produced, not copied at the end.
# Change the folder if your thesis results live elsewhere in Drive.
OUTDIR = "/content/drive/MyDrive/thesis/document_eval"
os.makedirs(OUTDIR, exist_ok=True)
print("writing results to:", OUTDIR)

## 3. Install dependencies

`easse` provides SARI/FKGL and installs from git (it is not on PyPI). The rest are the
metric and model libraries the script imports.

In [ ]:
%pip install -q transformers accelerate sacrebleu bert-score nltk
%pip install -q "git+https://github.com/feralvam/easse.git@master#egg=easse"

import nltk

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)  # nltk >= 3.8.2 uses punkt_tab for sent_tokenize
print("dependencies ready")

### 3b. LENS (optional)

LENS is skipped locally because `lens-metric` pins its own torch/transformers and would
conflict with the backend's versions. A throwaway Colab runtime has no such constraint, so
this is the one place LENS can actually be computed — which is worth doing, since it is the
only metric here trained on human simplification judgments.

**Run this cell before any model is loaded**, and restart the runtime if pip reports that it
changed `torch` or `transformers`. Skip the cell entirely to report D-SARI/BERTScore only;
the script degrades gracefully and records `"LENS": null` in the output.

In [ ]:
# Optional -- comment out the next line to skip LENS.
%pip install -q lens-metric

try:
    import lens

    print("LENS available")
except ImportError:
    print("LENS not installed -- the run will report LENS: null, which is fine.")

## 4. Get `evaluate_document.py`

Upload the file from `research/evaluate_document.py`. Once it is committed and pushed, the
`git clone` alternative below is the one-liner replacement.

In [ ]:
from google.colab import files

uploaded = files.upload()  # select research/evaluate_document.py
assert "evaluate_document.py" in uploaded, "Upload evaluate_document.py"
print("uploaded:", list(uploaded))

# Clone alternative, not usable yet: as of 2026-08-15 branch 2.0 (with evaluate_document.py)
# isn't pushed to origin. After `git push -u origin 2.0`, this replaces the upload above:
# !git clone --branch 2.0 --depth 1 https://github.com/yyvs/simple-website.git
# %cd simple-website/research

## 5. Download the D-Wikipedia test split

D-Wikipedia is not on the HF Hub — it ships as plain `.src`/`.tgt` text files (one document
per line, aligned by line number) in the authors' GitHub repo. Only the test split is needed
here, ~10 MB, and no 3.6 GB checkpoint upload is involved: the model comes from the Hub.

This one goes to the runtime's **local** disk, not Drive, and is re-downloaded after a
runtime death — it takes seconds, and keeping it off Drive avoids syncing 10 MB of static
corpus you already have. The file is byte-identical every time, which is what lets a
checkpoint written in an earlier session still match this session's document list.

In [ ]:
import os, urllib.request

BASE = "https://raw.githubusercontent.com/RLSNLP/Document-level-text-simplification/main/Dataset"
os.makedirs("scratch/d_wikipedia_raw", exist_ok=True)

for name in ("test.src", "test.tgt"):
    dest = f"scratch/d_wikipedia_raw/{name}"
    if not os.path.exists(dest):
        print("downloading", name)
        urllib.request.urlretrieve(f"{BASE}/{name}", dest)

for name in ("test.src", "test.tgt"):
    path = f"scratch/d_wikipedia_raw/{name}"
    with open(path, encoding="utf-8") as f:
        lines = f.read().splitlines()
    print(f"{name}: {len(lines)} documents, {os.path.getsize(path)/1e6:.1f} MB")

## 6. Run the evaluation

The model is pulled from the Hub, so nothing needs uploading.

**On `--limit`:** the default 500 is a seeded random sample of the cleaned split (the seed
and a fingerprint of the sampled indices go into the output, so the subset is reproducible).
Start there and confirm the throughput the log reports before considering `--limit 0`, which
evaluates all 8,000 documents. The generations are cached to `--outdir`, so re-scoring or
adding a metric later never regenerates.

Sampling is honest for the thesis as long as n and the seed are reported — both are in the
JSON — but the full split is preferable if the runtime allows it.

**If it dies partway, rerun the same cell.** Generation — the expensive part — resumes from
the last checkpoint (`partial_finetuned_*.json` / `partial_baseline_*.json` in `OUTDIR`),
skipping the documents already done. Keep `--limit` and `--seed` identical: a checkpoint is
bound to a fingerprint of the exact document list, and a mismatched one is discarded rather
than silently resumed from.

**Scoring is not resumed — only generation is.** If a run dies *during scoring*, the rerun
reloads the finished generations from the checkpoints and rescores from the top (a few
minutes, mostly BERTScore pulling roberta-large). The staged `run.status` in the result JSON
exists so you can see how far a killed run got; it is not a resume point.

**After a runtime death, rerun the setup cells first** — §2 (Drive), §3 (install), §4
(upload the script), §5 (download the corpus). Colab wipes local disk, so the script and the
corpus are gone even though the checkpoints on Drive are not. That is ~2 minutes against
hours of saved generation. Skip §6a — the smoke run proves nothing new once it has passed.

### 6a. Smoke run first (20 documents, ~1 minute)

The script's own docstring recommends this and it is worth the minute: it exercises every
step of the path — `easse` import, Hub download, batched generation, D-SARI, the paired
bootstrap, the JSON write — on a sample small enough to fail fast.

`easse` is the fragile link (installed from git, unmaintained, no PyPI release), and it is
imported at *scoring* time, i.e. after generation. On a 500-document run an install problem
would surface 30 minutes in. Here it surfaces immediately.

Output goes to `results/smoke` so it never mixes with the reportable run's artifacts.
If this cell prints a score block and a p-value, the real run below is safe to start.

In [ ]:
!python evaluate_document.py \
    --model yunvs/bart-base-dwikipedia-simplification \
    --baseline facebook/bart-base \
    --limit 20 \
    --batch-size 8 \
    --device cuda \
    --resamples 100 \
    --outdir {OUTDIR}/smoke

In [ ]:
!python evaluate_document.py \
    --model yunvs/bart-base-dwikipedia-simplification \
    --baseline facebook/bart-base \
    --limit 500 \
    --batch-size 16 \
    --device cuda \
    --resamples 1000 \
    --outdir {OUTDIR}

## 7. Inspect the result

The JSON is the artifact — `RESULTS.md` currently has "❌ none" in the Artifact column for
every row, and this is the first number that will not.

In [ ]:
import glob, json, os

path = sorted(glob.glob(f"{OUTDIR}/step4b_dwikipedia_*.json"))[-1]
result = json.load(open(path))
status = result["run"].get("status", "complete")

print(f"=== {os.path.basename(path)} ===\n")
print("status:   ", status, "" if status == "complete" else "  <-- INTERRUPTED")
print("model:    ", result["models"]["finetuned"]["model_id"])
print("baseline: ", result["models"]["baseline"]["model_id"])
print(
    "dataset:  ",
    result["dataset"]["name"],
    result["dataset"]["split"],
    f"n={result['dataset']['n_evaluated']}",
    (
        f"(seed {result['dataset']['sample_seed']})"
        if result["dataset"]["sampled"]
        else "(full split)"
    ),
)
print(
    "device:   ",
    result["run"]["device"],
    "| duration:",
    result["run"]["duration_seconds"],
    "s",
)
print("finished: ", result["run"]["finished_at"])
print()

scores = result["scores"]
fmt = lambda v: f"{v:>12.2f}" if isinstance(v, (int, float)) else f"{'--':>12}"
print(f"{'metric':<12}{'fine-tuned':>12}{'baseline':>12}")
for m in ["D-SARI", "SARI", "BLEU", "FKGL", "BERTScore", "LENS"]:
    print(
        f"{m:<12}{fmt(scores.get('finetuned', {}).get(m))}{fmt(scores.get('baseline', {}).get(m))}"
    )

sig = result.get("significance")
print(
    f"\nD-SARI significance: p = {sig['p_value']:.5f} ({sig['resamples']} resamples)"
    if sig
    else "\nD-SARI significance: not reached"
)

if status != "complete":
    print(
        f"\nThis run stopped after '{status}'. Rerun the evaluation cell with the same"
        f" --limit and --seed to pick up where it left off."
    )

In [ ]:
# Rescue anything written to local disk instead of Drive (e.g. a run started before OUTDIR
# existed). Per file, not zipped, so the small JSONs stay readable in Drive.
import glob, os, shutil

rescued = 0
for local in glob.glob("results/**/*.json", recursive=True):
    dest = os.path.join(OUTDIR, "rescued_from_local", os.path.relpath(local, "results"))
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    shutil.copy2(local, dest)
    print("copied", local, "->", dest)
    rescued += 1

print(
    f"\n{rescued} file(s) copied into Drive"
    if rescued
    else "\nnothing on local disk to rescue -- everything already went straight to Drive"
)

# Optional; only works while the tab is open:
# from google.colab import files
# files.download(path)

## 8. Back in the repo

Everything is already in `OUTDIR` on Drive — nothing to download before closing the tab.

1. Copy the result JSON out of Drive into `research/results/document_eval/` and commit it.
2. Fill row **D2** in `research/results/RESULTS.md` — D-SARI is the headline; plain SARI is
   there only for continuity with the ASSET numbers.
3. Update the thesis §6.2's "Document-Mode Results", which currently states that no
   document-level result exists. It will then be the first row in the whole results table
   backed by a saved artifact.
4. Keep the caveat that this is the **reduced-scope** checkpoint (20k documents, 2 epochs) —
   a document-level result, but not the full-scope one.

The cached generations are also the raw material for §6.3's qualitative error analysis,
which is still an open 🟠 section.